# RQ1: trích xuất feature và huấn luyện trên Colab

Notebook này chạy **12 nhóm train/validation đã khóa**, tạo learned features, kiểm tra receipts, huấn luyện C1–C6 và chấm validation. Nó không mở final test và không tạo kết quả official RQ1.

Colab hiện có thể cấp kernel Python 3.13. Notebook giữ kernel đó, nhưng tự tạo runtime pipeline Python 3.11 riêng vì PaddlePaddle GPU 2.6.2 chưa có wheel tương thích Python 3.13.

## Chuẩn bị một lần trên máy local

Tại nested repository `multimodal-lecture-summarizer`, chạy:

```powershell
python -m experiments.scripts.rq1_colab_bundle --output experiments/outputs/rq1_locked_mixed_20/rq1_colab_bundle_gpu_v3.zip
```

Script tạo `experiments/outputs/rq1_locked_mixed_20/rq1_colab_bundle_gpu_v3.zip` (khoảng 3 GiB). Tải file đó lên Google Drive tại `MyDrive/RQ1/rq1_colab_bundle_gpu_v3.zip`. Bundle chỉ chứa mã cần thiết, receipt D2, nguồn train/validation, các cache OCR hiện có và PaddleOCR weights; không chứa media final test.

Notebook sao chép bundle về ổ cục bộ Colab, tạo Python 3.11 bằng uv, cài stack đã khóa vào môi trường riêng, tải model revisions, chạy tối đa hai video song song, đồng bộ feature/cache lên Drive sau mỗi khoảng 45 giây và sau từng video. Nếu Colab cấp runtime mới sau khi bị ngắt, chạy lại notebook từ đầu; feature và cache đã đồng bộ sẽ được khôi phục ở bước 6.

Bản GPU v3 chạy lại từ đầu trong thư mục kết quả riêng. Tối đa hai video khác nhau chạy song song; không nhập tensor hay cache OCR của lượt GPU v2. Nếu Colab bị ngắt, chạy lại chính notebook này để tiếp tục từ checkpoint GPU v3.


## 1. Kết nối Drive và lấy bundle

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/RQ1')
BUNDLE_ON_DRIVE = DRIVE_ROOT / 'rq1_colab_bundle_gpu_v3.zip'
LOCAL_BUNDLE = Path('/content/rq1_colab_bundle_gpu_v3.zip')
if not BUNDLE_ON_DRIVE.is_file():
    raise FileNotFoundError(f'Upload bundle first: {BUNDLE_ON_DRIVE}')
shutil.copy2(BUNDLE_ON_DRIVE, LOCAL_BUNDLE)
print(f'Bundle copied to Colab local disk: {LOCAL_BUNDLE} ({LOCAL_BUNDLE.stat().st_size / 2**30:.2f} GiB)')


## 2. Giải nén mã và dữ liệu vào ổ cục bộ Colab

In [ ]:
import zipfile
import hashlib
import os

WORK_ROOT = Path('/content/rq1_work')
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

bundle_sha256 = sha256_file(LOCAL_BUNDLE)
with zipfile.ZipFile(LOCAL_BUNDLE) as archive:
    for member in archive.infolist():
        target = (WORK_ROOT / member.filename).resolve()
        if WORK_ROOT.resolve() not in target.parents and target != WORK_ROOT.resolve():
            raise ValueError(f'Unsafe archive path: {member.filename}')
    archive.extractall(WORK_ROOT)

REPO = WORK_ROOT / 'multimodal-lecture-summarizer'
if not (REPO / 'experiments/scripts/rq1_safe_resume.py').is_file():
    raise FileNotFoundError('The bundle does not contain the RQ1 runner')
os.chdir(REPO)
OUT_ROOT = REPO / 'experiments/outputs/rq1_locked_mixed_20'
FEATURE_OUT = OUT_ROOT / 'learned_features_colab_gpu_v3'
TRAIN_OUT = OUT_ROOT / 'learned_training_colab_gpu_v3'
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_FEATURES = DRIVE_RESULTS / 'learned_features_colab_gpu_v3'
DRIVE_TRAINING = DRIVE_RESULTS / 'learned_training_colab_gpu_v3'
print('Project root:', REPO)
print('Bundle SHA256:', bundle_sha256)


## 3. Kiểm tra runtime trước khi cài model

In [ ]:
import subprocess
import psutil
import sys

# Do not gate on the Colab kernel version; the pinned pipeline runs in its own Python 3.11 venv.
subprocess.run(['nvidia-smi'], check=True)
ram_available_gib = psutil.virtual_memory().available / 2**30
print(f'Colab kernel Python: {sys.version}')
print(f'RAM available: {ram_available_gib:.2f} GiB')
if ram_available_gib < 6:
    raise RuntimeError('Available RAM is below 6 GiB. Restart with a higher-RAM runtime before proceeding.')


## 4. Cài đúng các phiên bản đã ghi trong feature receipt

In [ ]:
# Use uv-managed CPython 3.11 for the pinned Paddle GPU wheels, regardless of Colab kernel version.
%pip install -q uv
import subprocess
import sys
from pathlib import Path

VENV = Path('/content/rq1-py311')
PYTHON_BIN = str(VENV / 'bin' / 'python')
subprocess.run([sys.executable, '-m', 'uv', 'python', 'install', '3.11'], check=True)

def is_usable_py311(path):
    if not Path(path).is_file():
        return False
    probe = subprocess.run(
        [path, '-c', "import sys; print(sys.version_info[:2]); assert sys.version_info[:2] == (3, 11)"],
        capture_output=True, text=True)
    return probe.returncode == 0

if is_usable_py311(PYTHON_BIN):
    print('Reusing existing pipeline Python 3.11 environment.')
else:
    # Recover from an interrupted/partial environment left by an earlier cell run.
    if VENV.exists():
        import shutil
        if VENV.is_symlink() or VENV.resolve().parent != Path('/content').resolve():
            raise RuntimeError(f'Refusing to replace unexpected environment path: {VENV}')
        shutil.rmtree(VENV)
    create = subprocess.run(
        [sys.executable, '-m', 'uv', 'venv', '--seed', '--python', '3.11', str(VENV)],
        capture_output=True, text=True)
    if create.stdout:
        print(create.stdout, end='')
    if create.stderr:
        print(create.stderr, end='', file=sys.stderr)
    if create.returncode:
        raise RuntimeError(f'uv venv failed with exit code {create.returncode}; see output above')

if not is_usable_py311(PYTHON_BIN):
    raise RuntimeError(f'Pipeline Python 3.11 environment is unusable: {PYTHON_BIN}')
print('Pipeline interpreter:', subprocess.check_output([PYTHON_BIN, '--version'], text=True).strip())

# safe_resume resolves its OCR interpreter path. Keep that path as a regular wrapper
# so resolving it does not bypass the venv's site-packages through uv's Python symlink.
OCR_PYTHON = str(VENV / 'bin' / 'python-ocr')
Path(OCR_PYTHON).write_text(
    f'#!/bin/sh\nexec "{PYTHON_BIN}" "$@"\n', encoding='utf-8')
Path(OCR_PYTHON).chmod(0o755)

# Official PyTorch CUDA 12.4 install command, targeting the isolated Python 3.11 venv.
# Use pip's CUDA index resolution exactly as published, without mixing in a second index.
subprocess.run([PYTHON_BIN, '-m', 'pip', 'install', '--no-cache-dir',
                'torch==2.6.0', 'torchvision==0.21.0',
                '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)

# Scientific/vision stack pinned to the existing RQ1 receipts.
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON_BIN,
                'numpy==2.4.6', 'scipy==1.17.1', 'transformers==4.57.6',
                'sentence-transformers==5.7.0', 'opencv-python==4.11.0.86', 'psutil'], check=True)

# Existing OCR contract is PaddleOCR 2.8.1 + PaddlePaddle GPU 2.6.2 / CUDA 11.8.
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON_BIN,
                '--find-links', 'https://www.paddlepaddle.org.cn/packages/stable/cu118/',
                'paddlepaddle-gpu==2.6.2'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON_BIN,
                'paddleocr==2.8.1'], check=True)
subprocess.run(['bash', '-lc', 'apt-get -qq update && apt-get -qq install -y ffmpeg'], check=True)


## 5. Xác nhận GPU và tải model revisions đã khóa

In [ ]:
import json
import subprocess
import sys
from experiments.scripts.rq1_learned_features import (
    TEXT_MODEL, TEXT_REVISION, VISUAL_MODEL, VISUAL_REVISION,
)

# Keep Paddle and PyTorch in separate child processes, matching the OCR isolation contract.
torch_probe = subprocess.run(
    [PYTHON_BIN, '-c', "import torch; print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"],
    check=True, capture_output=True, text=True)
torch_lines = torch_probe.stdout.strip().splitlines()
if not torch_lines or torch_lines[0] != 'True':
    raise RuntimeError('PyTorch cannot access a CUDA GPU. Select a Colab GPU runtime.')
GPU_NAME = torch_lines[1]
paddle_probe = subprocess.run(
    [PYTHON_BIN, '-c', "import paddle; print(paddle.is_compiled_with_cuda()); print(paddle.device.cuda.device_count())"],
    check=True, capture_output=True, text=True)
paddle_lines = paddle_probe.stdout.strip().splitlines()
if not paddle_lines or paddle_lines[0] != 'True' or int(paddle_lines[1]) < 1:
    raise RuntimeError('Paddle GPU is unavailable; do not continue with mixed OCR compute.')
version_probe = subprocess.run(
    [PYTHON_BIN, '-c', "import importlib.metadata as m, json; print(json.dumps({n:m.version(n) for n in ['torch','paddleocr','paddlepaddle-gpu']}))"],
    check=True, capture_output=True, text=True)
print('GPU:', GPU_NAME)
print('Pipeline versions:', version_probe.stdout.strip())

MODEL_CACHE = OUT_ROOT / 'model_cache'
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
download_code = "from huggingface_hub import snapshot_download; import sys; snapshot_download(repo_id=sys.argv[1], revision=sys.argv[2], cache_dir=sys.argv[3])"
for model, revision in ((TEXT_MODEL, TEXT_REVISION), (VISUAL_MODEL, VISUAL_REVISION)):
    subprocess.run([PYTHON_BIN, '-c', download_code, model, revision, str(MODEL_CACHE)], check=True)
print('Pinned MiniLM and DINOv2 snapshots are cached on Colab local disk.')


## 6. Khôi phục kết quả trước đó và chuẩn bị cache OCR

In [ ]:
import shutil


def sync_tree(source, destination):
    source, destination = Path(source), Path(destination)
    if not source.is_dir():
        return
    for path in source.rglob('*'):
        if not path.is_file() or path.name.endswith('.partial'):
            continue
        target = destination / path.relative_to(source)
        target.parent.mkdir(parents=True, exist_ok=True)
        if (not target.is_file() or target.stat().st_size != path.stat().st_size
                or path.stat().st_mtime > target.stat().st_mtime):
            shutil.copy2(path, target)

# Restore only GPU v3 checkpoints if this fresh run was interrupted.
sync_tree(DRIVE_FEATURES, FEATURE_OUT)

from experiments.scripts.rq1_learned_features import development_sources
rows = development_sources()
assert len(rows) == 12 and {row['split'] for row in rows} == {'train', 'validation'}
assert sum(row['split'] == 'train' for row in rows) == 8
print(f'Locked development units: {len(rows)} (8 train, 4 validation)')
print('Feature output:', FEATURE_OUT)
print('Drive checkpoint folder:', DRIVE_FEATURES)


## 7. Trích xuất từng video, đồng bộ định kỳ lên Drive

In [ ]:
import queue
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import time


SYNC_LOCK = threading.Lock()

def sync_with_heartbeat(source, destination):
    # Two unit workers may finish at once; serialize copies into the same Drive folder.
    with SYNC_LOCK:
        sync_tree(source, destination)


def run_and_sync(command, source, destination, interval_sec=45, cpu_threads=4):
    print('RUN:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1,
                               env={**os.environ, 'HF_HUB_OFFLINE': '1',
                                    'TRANSFORMERS_OFFLINE': '1',
                                    'HF_HUB_DISABLE_TELEMETRY': '1',
                                    'PYTHONUNBUFFERED': '1', 'RQ1_CPU_THREADS': str(cpu_threads)})
    output_queue = queue.Queue()

    def collect():
        for line in process.stdout:
            output_queue.put(line)
        output_queue.put(None)

    reader = threading.Thread(target=collect, daemon=True)
    reader.start()
    last_sync = time.monotonic()
    ended = False
    try:
        while not ended or process.poll() is None:
            try:
                line = output_queue.get(timeout=1)
                if line is None:
                    ended = True
                else:
                    print(line, end='', flush=True)
            except queue.Empty:
                pass
            if time.monotonic() - last_sync >= interval_sec:
                sync_with_heartbeat(source, destination)
                print('Drive checkpoint synchronized.', flush=True)
                last_sync = time.monotonic()
        code = process.wait()
    finally:
        if process.poll() is None:
            stop_process_tree(process)
        sync_with_heartbeat(source, destination)
        print('Drive checkpoint synchronized after process exit.', flush=True)
    if code != 0:
        raise RuntimeError(f'RQ1 child command failed with exit code {code}; local and Drive caches were synchronized.')

from experiments.scripts.rq1_safe_resume import gpu_memory_mib, stop_process_tree
_, _, GPU_TOTAL_MIB = gpu_memory_mib()
MAX_GPU_USED_MIB = int(GPU_TOTAL_MIB * 0.85)
MIN_GPU_FREE_MIB = max(1024, int(GPU_TOTAL_MIB * 0.10))
VISUAL_BATCH_SIZE = 16
print(f'GPU guard: used < {MAX_GPU_USED_MIB} MiB, free >= {MIN_GPU_FREE_MIB} MiB')

MAX_PARALLEL_UNITS = 2
pending_rows = sorted(
    (row for row in rows
     if not ((FEATURE_OUT / f"{row['unit_id']}.pt").is_file()
             and (FEATURE_OUT / f"{row['unit_id']}.receipt.json").is_file())),
    key=lambda row: (row['duration_sec'], row['unit_id']))
print(f'Already complete: {len(rows) - len(pending_rows)}/12; pending: {len(pending_rows)}')

def run_unit(row):
    # Per-unit isolation avoids re-verifying all earlier units on every resume call.
    command = [PYTHON_BIN, '-m', 'experiments.scripts.rq1_safe_resume',
               '--output', str(FEATURE_OUT), '--ocr-python', OCR_PYTHON,
               f"--unit-id={row['unit_id']}", '--visual-device', 'cuda',
               '--visual-batch-size', str(VISUAL_BATCH_SIZE),
               '--max-gpu-used-mib', str(MAX_GPU_USED_MIB),
               '--min-gpu-free-mib', str(MIN_GPU_FREE_MIB)]
    run_and_sync(command, FEATURE_OUT, DRIVE_FEATURES, cpu_threads=1)
    return row['unit_id']

with ThreadPoolExecutor(max_workers=MAX_PARALLEL_UNITS) as pool:
    futures = {pool.submit(run_unit, row): row['unit_id'] for row in pending_rows}
    try:
        for future in as_completed(futures):
            print('Completed and synchronized:', future.result(), flush=True)
    except BaseException:
        for future in futures:
            future.cancel()
        raise

assert all((FEATURE_OUT / f"{row['unit_id']}.pt").is_file()
           and (FEATURE_OUT / f"{row['unit_id']}.receipt.json").is_file()
           for row in rows)
print('All 12 train/validation feature tensors have been written.')


## 8. Xác minh đủ 12 tensor rồi huấn luyện và chấm validation

In [ ]:
verify = [PYTHON_BIN, '-m', 'experiments.scripts.rq1_learned_features',
          '--output', str(FEATURE_OUT), '--verify']
run_and_sync(verify, FEATURE_OUT, DRIVE_FEATURES)

train = [PYTHON_BIN, '-m', 'experiments.scripts.rq1_train_learned',
         '--feature-dir', str(FEATURE_OUT), '--output', str(TRAIN_OUT),
         '--device', 'cuda', '--preload-features']
sync_tree(DRIVE_TRAINING, TRAIN_OUT)
run_and_sync(train, TRAIN_OUT, DRIVE_TRAINING, interval_sec=45)
print('Training and validation scoring completed. Final-test data was not accessed.')


## 9. Trực quan hóa kết quả validation trước khi lưu manifest


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

result_dir = TRAIN_OUT / 'full'
scores = json.loads((result_dir / 'validation_scores.json').read_text(encoding='utf-8'))
selection = json.loads((result_dir / 'selected.json').read_text(encoding='utf-8'))
candidates = [f'C{i}' for i in range(1, 7)]
unit_ids = selection['validation_unit_ids']
assert len(unit_ids) == 4 and len(scores) == 24
assert selection['final_test_accessed'] is False
assert {(r['candidate'], r['unit_id']) for r in scores} == {
    (candidate, uid) for candidate in candidates for uid in unit_ids
}
assert all(r['status'] == 'run' for r in scores)

by_key = {(r['candidate'], r['unit_id']): r for r in scores}
f1 = np.array([[by_key[c, u]['metrics']['collar_f1_3s'] for u in unit_ids]
               for c in candidates], dtype=float)
predicted = np.array([[by_key[c, u]['metrics']['predicted_boundary_count']
                       for u in unit_ids] for c in candidates], dtype=float)
reference = np.array([by_key['C1', u]['metrics']['reference_boundary_count']
                      for u in unit_ids], dtype=float)
assert np.isfinite(f1).all() and np.isfinite(predicted).all()
assert all(by_key[c, u]['metrics']['reference_boundary_count'] == reference[j]
           for c in candidates for j, u in enumerate(unit_ids))

fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
f1_image = axes[0].imshow(f1, vmin=0, vmax=1, cmap='Blues', aspect='auto')
fig.colorbar(f1_image, ax=axes[0], label='F1@3s')
axes[0].set_title('Validation: F1@±3s')
for i in range(len(candidates)):
    for j in range(len(unit_ids)):
        axes[0].text(j, i, f'{f1[i, j]:.3f}', ha='center', va='center',
                     color='white' if f1[i, j] > 0.55 else 'black')
axes[0].set_xticks(range(len(unit_ids)), unit_ids, rotation=30, ha='right')
axes[0].set_yticks(range(len(candidates)), candidates)
axes[0].set_xlabel('Validation video')

count_image = axes[1].imshow(predicted, vmin=0,
                             vmax=max(1, float(np.max(predicted)), float(np.max(reference))),
                             cmap='Oranges', aspect='auto')
fig.colorbar(count_image, ax=axes[1], label='Predicted boundaries')
axes[1].set_title('Predicted boundaries; GT count under each video')
for i in range(len(candidates)):
    for j in range(len(unit_ids)):
        axes[1].text(j, i, f'{predicted[i, j]:.0f}', ha='center', va='center')
axes[1].set_xticks(range(len(unit_ids)),
                   [f'{u}\nGT={reference[j]:.0f}' for j, u in enumerate(unit_ids)],
                   rotation=30, ha='right')
axes[1].set_yticks(range(len(candidates)), candidates)
axes[1].set_xlabel('Validation video')

display(fig)
plot_path = result_dir / 'validation_overview.png'
fig.savefig(plot_path, dpi=160, bbox_inches='tight')
plt.close(fig)
print('Mean validation F1@3s:')
for i, candidate in enumerate(candidates):
    print(f'  {candidate}: {f1[i].mean():.4f}; predicted={predicted[i].sum():.0f}; '
          f'GT={reference.sum():.0f}; seed={selection["selected"][candidate]["seed"]}')
print('Plot:', plot_path)


## 10. Lưu manifest chạy và đồng bộ kết quả

In [ ]:
import json

versions_result = subprocess.run(
    [PYTHON_BIN, '-c', "import importlib.metadata as m, json; print(json.dumps({n:m.version(n) for n in ['torch','transformers','sentence-transformers','opencv-python','scipy','numpy','paddleocr','paddlepaddle-gpu']}))"],
    check=True, capture_output=True, text=True)
versions = json.loads(versions_result.stdout)
pipeline_python = subprocess.check_output([PYTHON_BIN, '-c', 'import platform; print(platform.python_version())'], text=True).strip()
run_record = {
    'schema': 'rq1-colab-run/1',
    'bundle_sha256': bundle_sha256,
    'colab_kernel_python': sys.version.split()[0],
    'pipeline_python': pipeline_python,
    'gpu': GPU_NAME,
    'visual_compute': 'cuda',
    'visual_batch_size': VISUAL_BATCH_SIZE,
    'training_compute': 'cuda',
    'training_features_preloaded': True,
    'parallel_feature_units': MAX_PARALLEL_UNITS,
    'cpu_threads_per_feature_unit': 1,
    'gpu_guard': {'max_gpu_used_mib': MAX_GPU_USED_MIB,
                  'min_gpu_free_mib': MIN_GPU_FREE_MIB},
    'versions': versions,
    'feature_output': str(DRIVE_FEATURES),
    'training_output': str(DRIVE_TRAINING),
    'validation_visualization': str(DRIVE_TRAINING / 'full' / 'validation_overview.png'),
    'feature_units': [row['unit_id'] for row in rows],
    'split_counts': {'train': 8, 'validation': 4, 'final_test_accessed': False},
}
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
(DRIVE_RESULTS / 'rq1_colab_run_gpu_v3_fresh.json').write_text(json.dumps(run_record, indent=2) + '\n', encoding='utf-8')
sync_tree(FEATURE_OUT, DRIVE_FEATURES)
sync_tree(TRAIN_OUT, DRIVE_TRAINING)
print(json.dumps(run_record, indent=2))
print('Saved run record:', DRIVE_RESULTS / 'rq1_colab_run_gpu_v3_fresh.json')
